In [88]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    KFold
)

In [72]:
df = pd.read_csv('zomato.csv')

In [73]:
df.head()
# df.columns.to_list()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [74]:

df.shape

(51717, 17)

In [75]:
df.isnull().sum()

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            7775
votes                              0
phone                           1208
location                          21
rest_type                        227
dish_liked                     28078
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

In [76]:
df.duplicated(subset=['name', 'address','listed_in(type)']).sum()

np.int64(30802)

In [53]:
df.duplicated().sum()

np.int64(0)

In [70]:
df.duplicated(subset=['name']).sum()

np.int64(42925)

In [71]:
df.duplicated(subset=['name', 'address']).sum()


np.int64(39218)

In [72]:
df['listed_in(type)'].value_counts()

listed_in(type)
Delivery              25942
Dine-out              17779
Desserts               3593
Cafes                  1723
Drinks & nightlife     1101
Buffet                  882
Pubs and bars           697
Name: count, dtype: int64

In [54]:
df[
    df.duplicated(
        subset=['name', 'address'],
        keep=False
    )
].sort_values(['name', 'address'])

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
4321,https://www.zomato.com/bangalore/feeltheroll-b...,"Opposite Mantri Commercio, Outer Ring Road, De...",#FeelTheROLL,No,No,3.4/5,7,+91 9108342079\r\n+91 9886117901,Bellandur,Quick Bites,NaN,Fast Food,200,"[('Rated 5.0', ""RATED\n Had an egg chicken ro...",[],Delivery,Bellandur
49511,https://www.zomato.com/bangalore/feeltheroll-b...,"Opposite Mantri Commercio, Outer Ring Road, De...",#FeelTheROLL,No,No,3.4 /5,7,+91 9108342079\n+91 9886117901,Bellandur,Quick Bites,NaN,Fast Food,200,"[('Rated 5.0', ""RATED\n Had an egg chicken ro...",[],Delivery,Sarjapur Road
10182,https://www.zomato.com/bangalore/l-81-cafe-hsr...,"Sector 6, HSR Layout, HSR",#L-81 Cafe,Yes,No,3.9/5,48,+91 9986210891,HSR,Quick Bites,Burgers,"Fast Food, Beverages",400,"[('Rated 4.0', 'RATED\n This little cafe is s...",[],Delivery,BTM
11096,https://www.zomato.com/bangalore/l-81-cafe-hsr...,"Sector 6, HSR Layout, HSR",#L-81 Cafe,Yes,No,3.9/5,48,+91 9986210891,HSR,Quick Bites,Burgers,"Fast Food, Beverages",400,"[('Rated 4.0', 'RATED\n This little cafe is s...",[],Dine-out,BTM
16606,https://www.zomato.com/bangalore/l-81-cafe-hsr...,"Sector 6, HSR Layout, HSR",#L-81 Cafe,Yes,No,3.9/5,48,+91 9986210891,HSR,Quick Bites,Burgers,"Fast Food, Beverages",400,"[('Rated 4.0', 'RATED\n This little cafe is s...",[],Delivery,HSR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9665,https://www.zomato.com/bangalore/scoolmeal-btm...,"197/293-3, 32nd Main, 1st Stage, BTM, Bangalore",sCoolMeal,No,No,NaN,0,+91 8971958284,BTM,"Takeaway, Delivery",NaN,"North Indian, South Indian, Chinese, Pizza",300,[],[],Delivery,BTM
27925,https://www.zomato.com/bangalore/scoolmeal-btm...,"197/293-3, 32nd Main, 1st Stage, BTM, Bangalore",sCoolMeal,No,No,NaN,0,+91 8971958284,BTM,"Takeaway, Delivery",NaN,"North Indian, South Indian, Chinese, Pizza",300,[],[],Delivery,Koramangala 4th Block
30405,https://www.zomato.com/bangalore/scoolmeal-btm...,"197/293-3, 32nd Main, 1st Stage, BTM, Bangalore",sCoolMeal,No,No,NaN,0,+91 8971958284,BTM,"Takeaway, Delivery",NaN,"North Indian, South Indian, Chinese, Pizza",300,[],[],Delivery,Koramangala 5th Block
33476,https://www.zomato.com/bangalore/scoolmeal-btm...,"197/293-3, 32nd Main, 1st Stage, BTM, Bangalore",sCoolMeal,No,No,NaN,0,+91 8971958284,BTM,"Takeaway, Delivery",NaN,"North Indian, South Indian, Chinese, Pizza",300,[],[],Delivery,Koramangala 6th Block


In [77]:
df['_rate_num'] = pd.to_numeric(
    df['rate']
    .astype(str)
    .str.extract(r'(\d+(?:\.\d+)?)')[0],
    errors='coerce'
)
before = len(df)
df = df.sort_values(
    by=['_rate_num', 'votes'],
    ascending=[False, False],
    na_position='last'
)
df = df.drop_duplicates(
    subset=['name', 'address', 'listed_in(type)'],
    keep='first'
)
df = df.drop(columns=['_rate_num']).reset_index(drop=True)
print("Rows before:", before)
print("Rows after:", len(df))
print("Duplicates removed:", before - len(df))

Rows before: 51717
Rows after: 20915
Duplicates removed: 30802


In [78]:
df.shape

(20915, 17)

In [79]:
df.duplicated(subset=['name', 'address','listed_in(type)']).sum()

np.int64(0)

In [80]:
df.isnull().sum()

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            3285
votes                              0
phone                            414
location                          12
rest_type                        107
dish_liked                     11951
cuisines                          25
approx_cost(for two people)      106
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

In [59]:
df['rate'].isnull().mean() * 100

np.float64(15.706430791298112)

In [81]:
print("Before:", df.shape[0])
df['rate'] = pd.to_numeric(
    df['rate'].astype(str).str.extract(r'(\d+(?:\.\d+)?)')[0],
    errors='coerce'
)
df = df.dropna(subset=['rate']).copy()
print("After:", df.shape[0])

Before: 20915
After: 16654


In [82]:
df.isnull().sum()

url                               0
address                           0
name                              0
online_order                      0
book_table                        0
rate                              0
votes                             0
phone                           279
location                          0
rest_type                        66
dish_liked                     7722
cuisines                          6
approx_cost(for two people)      71
reviews_list                      0
menu_item                         0
listed_in(type)                   0
listed_in(city)                   0
dtype: int64

In [83]:
cost_col = 'approx_cost(for two people)'

df[cost_col] = pd.to_numeric(
    df[cost_col].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)
df[['rate', 'votes', cost_col]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 16654 entries, 0 to 16653
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   rate                         16654 non-null  float64
 1   votes                        16654 non-null  int64  
 2   approx_cost(for two people)  16583 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 520.4 KB


In [84]:
numeric_cols = ['rate', 'votes', cost_col]
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'\nColumn: {col}')
    print('Lower bound:', lower)
    print('Upper bound:', upper)
    print('Outliers count:', len(outliers))
    print('Outliers percentage:',
          round(len(outliers) / len(df) * 100, 2), '%')


Column: rate
Lower bound: 2.5
Upper bound: 4.9
Outliers count: 82
Outliers percentage: 0.49 %

Column: votes
Lower bound: -312.5
Upper bound: 571.5
Outliers count: 2122
Outliers percentage: 12.74 %

Column: approx_cost(for two people)
Lower bound: -300.0
Upper bound: 1300.0
Outliers count: 1137
Outliers percentage: 6.83 %


In [16]:
df.shape

(16654, 17)

In [85]:
df = df.drop(columns=['phone', 'dish_liked'])
df = df.dropna(subset=['cuisines','rest_type'])

cost_col = 'approx_cost(for two people)'
df[cost_col] = pd.to_numeric(
    df[cost_col].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
) 
df[cost_col] = df[cost_col].fillna(
    df[cost_col].median()
)
print("Missing values:")
print(df.isnull().sum())
print("\nDataset shape:")
print(df.shape)


Missing values:
url                            0
address                        0
name                           0
online_order                   0
book_table                     0
rate                           0
votes                          0
location                       0
rest_type                      0
cuisines                       0
approx_cost(for two people)    0
reviews_list                   0
menu_item                      0
listed_in(type)                0
listed_in(city)                0
dtype: int64

Dataset shape:
(16582, 15)


In [65]:
df['location'].value_counts()

location
Whitefield              1098
HSR                      930
BTM                      900
Marathahalli             896
Indiranagar              869
                        ... 
Rajarajeshwari Nagar       2
West Bangalore             2
Central Bangalore          1
Nagarbhavi                 1
Peenya                     1
Name: count, Length: 92, dtype: int64

In [86]:
df_model = df.copy()
drop_cols = [
    'url',
    'name',
    'address',
    'reviews_list',
    'menu_item',
    'listed_in(city)'
]
df_model = df_model.drop(columns=drop_cols)
binary_cols = ['online_order', 'book_table']
for col in binary_cols:
    df_model[col] = (
        df_model[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({'yes': 1, 'no': 0})
    )
for col in ['rest_type', 'cuisines','location']:
    counts = df_model[col].value_counts()
    rare_categories = counts[counts < 50].index

    df_model[col] = df_model[col].replace(
        rare_categories, 'Other'
    )
categorical_cols = [
    'rest_type',
    'cuisines',
    'listed_in(type)',
    'location'
]

df_model = pd.get_dummies(
    df_model,
    columns=categorical_cols,
    dtype=int
)

# فصل الـ Features عن الـ Target
X = df_model.drop(columns=['rate'])
y = df_model['rate']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())


X shape: (16582, 157)
y shape: (16582,)
Missing values in X: 0
Missing values in y: 0


In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [68]:
models = {
    'Linear Regression': make_pipeline(
        StandardScaler(),
        LinearRegression()
    ),
    'Decision Tree': DecisionTreeRegressor(
        random_state=42,
        min_samples_leaf=5
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ),
    'KNN': make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=5)
    ),
    'SVR': make_pipeline(
        StandardScaler(),
        SVR()
    )
}

In [69]:
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Train R2': r2_score(y_train, train_pred),
        'Test R2': r2_score(y_test, test_pred),
        'Train MAE': mean_absolute_error(y_train, train_pred),
        'Test MAE': mean_absolute_error(y_test, test_pred),
        'Train RMSE': mean_squared_error(
            y_train, train_pred
        ) ** 0.5,
        'Test RMSE': mean_squared_error(
            y_test, test_pred
        ) ** 0.5
    })
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by='Test R2',
    ascending=False
).reset_index(drop=True)
print("\nModel Comparison:")
print(results_df.round(4).to_string(index=False))



Model Comparison:
            Model  Train R2  Test R2  Train MAE  Test MAE  Train RMSE  Test RMSE
    Random Forest    0.8911   0.6316     0.0964    0.1812      0.1452     0.2647
    Decision Tree    0.7385   0.4607     0.1542    0.2245      0.2249     0.3203
              SVR    0.5876   0.4368     0.1973    0.2405      0.2825     0.3273
Linear Regression    0.3735   0.3627     0.2712    0.2701      0.3482     0.3482
              KNN    0.5429   0.3067     0.2277    0.2794      0.2974     0.3631


In [90]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [91]:
searches = {
    "Decision Tree": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth": [None, 5, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [2, 5, 10, 15],
            "max_features": [1.0, "sqrt", 0.8]
        }
    ),

    "Random Forest": (
        RandomForestRegressor(
            random_state=42,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200, 300, 500],
            "max_depth": [None, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 3, 5, 8],
            "max_features": [1.0, "sqrt", 0.8]
        }
    ),

    "KNN": (
        make_pipeline(
            StandardScaler(),
            KNeighborsRegressor()
        ),
        {
            "kneighborsregressor__n_neighbors": [
                3, 5, 7, 10, 15, 20, 30
            ],
            "kneighborsregressor__weights": [
                "uniform", "distance"
            ],
            "kneighborsregressor__p": [1, 2]
        }
    ),

    "SVR": (
        make_pipeline(
            StandardScaler(),
            SVR()
        ),
        {
            "svr__kernel": ["rbf"],
            "svr__C": [0.1, 1, 10, 50, 100],
            "svr__epsilon": [0.05, 0.1, 0.2, 0.3],
            "svr__gamma": ["scale", "auto"]
        }
    )
}

In [92]:
best_models = {}
results = []
tuning_results = []
# Linear Regression as a baseline
baseline = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

baseline.fit(X_train, y_train)

baseline_train_pred = baseline.predict(X_train)
baseline_test_pred = baseline.predict(X_test)

results.append({
    "Model": "Linear Regression",
    "Train R2": r2_score(y_train, baseline_train_pred),
    "Test R2": r2_score(y_test, baseline_test_pred),
    "Train MAE": mean_absolute_error(
        y_train, baseline_train_pred
    ),
    "Test MAE": mean_absolute_error(
        y_test, baseline_test_pred
    ),
    "Train RMSE": np.sqrt(
        mean_squared_error(y_train, baseline_train_pred)
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(y_test, baseline_test_pred)
    )
})

best_models["Linear Regression"] = baseline

for name, (model, param_grid) in searches.items():

    print(f"\nTuning {name}...")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=20,
        scoring="neg_mean_absolute_error",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    best_models[name] = best_model

    train_pred = best_model.predict(X_train)
    test_pred = best_model.predict(X_test)

    results.append({
        "Model": name,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
        "Train MAE": mean_absolute_error(
            y_train, train_pred
        ),
        "Test MAE": mean_absolute_error(
            y_test, test_pred
        ),
        "Train RMSE": np.sqrt(
            mean_squared_error(y_train, train_pred)
        ),
        "Test RMSE": np.sqrt(
            mean_squared_error(y_test, test_pred)
        )
    })

    tuning_results.append({
        "Model": name,
        "Best CV MAE": -search.best_score_,
        "Best Parameters": search.best_params_
    })

    print("Best CV MAE:", round(-search.best_score_, 4))
    print("Best parameters:", search.best_params_)




Tuning Decision Tree...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV MAE: 0.2151
Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 20}

Tuning Random Forest...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV MAE: 0.1721
Best parameters: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}

Tuning KNN...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV MAE: 0.2838
Best parameters: {'kneighborsregressor__weights': 'uniform', 'kneighborsregressor__p': 1, 'kneighborsregressor__n_neighbors': 5}

Tuning SVR...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV MAE: 0.2461
Best parameters: {'svr__kernel': 'rbf', 'svr__gamma': 'auto', 'svr__epsilon': 0.05, 'svr__C': 1}


In [93]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test MAE"
).reset_index(drop=True)

print("\n" + "=" * 85)
print("MODEL COMPARISON AFTER TUNING")
print("=" * 85)

print(results_df.round(4).to_string(index=False))


MODEL COMPARISON AFTER TUNING
            Model  Train R2  Test R2  Train MAE  Test MAE  Train RMSE  Test RMSE
    Random Forest    0.9577   0.6991     0.0587    0.1574      0.0905     0.2392
    Decision Tree    0.7656   0.4904     0.1264    0.2041      0.2130     0.3113
              SVR    0.5888   0.4321     0.1853    0.2370      0.2821     0.3287
Linear Regression    0.3735   0.3627     0.2712    0.2701      0.3482     0.3482
              KNN    0.5595   0.3261     0.2229    0.2752      0.2920     0.3580


In [94]:
tuning_df = pd.DataFrame(tuning_results)

print("\n" + "=" * 85)
print("BEST CROSS-VALIDATION RESULTS AND PARAMETERS")
print("=" * 85)

if not tuning_df.empty:
    print(tuning_df.to_string(index=False))



BEST CROSS-VALIDATION RESULTS AND PARAMETERS
        Model  Best CV MAE                                                                                                 Best Parameters
Decision Tree     0.215078                           {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 20}
Random Forest     0.172146    {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}
          KNN     0.283777 {'kneighborsregressor__weights': 'uniform', 'kneighborsregressor__p': 1, 'kneighborsregressor__n_neighbors': 5}
          SVR     0.246055                                 {'svr__kernel': 'rbf', 'svr__gamma': 'auto', 'svr__epsilon': 0.05, 'svr__C': 1}


In [95]:
best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]
best_row = results_df.iloc[0]

print("\n" + "=" * 60)
print("BEST MODEL AFTER TUNING")
print("=" * 60)

print("Best model:", best_model_name)
print("Test R2:", round(best_row["Test R2"], 4))
print("Test MAE:", round(best_row["Test MAE"], 4))
print("Test RMSE:", round(best_row["Test RMSE"], 4))


BEST MODEL AFTER TUNING
Best model: Random Forest
Test R2: 0.6991
Test MAE: 0.1574
Test RMSE: 0.2392
